In [0]:
%run /Workspace/Users/gustavosousa.md20@gmail.com/retreino_desloc_databricks/00_struct_table

In [0]:
df = spark.table("tbl_ml").toPandas()

# Não recomendado
# df.drop(columns=["periodo"], inplace=True)

In [0]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["vlr_pago"])
Y = df["vlr_pago"]

X_train, X_test, y_train, y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42
)

In [0]:
colunas_minmax = [
    'dia',
    'dia_semana',
    'mes',
    'hora',
    'distancia',
    'latitude_origem',
    'longitude_origem',
    'latitude_destino',
    'longitude_destino'
]

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler

# Desabilitar o MLFLOW automatico do Databricks
import mlflow
mlflow.autolog(disable=True)

# Definir os processadores
preprocessador = ColumnTransformer(
    transformers=[
        ('drop', 'drop', ['periodo']),
        ('minmax', MinMaxScaler(), colunas_minmax)
    ],
    remainder='passthrough'  # mantém as outras colunas sem alteração
)

# Combinar os processadores + pipeline
pipeline = Pipeline(
    steps=[
        ('preprocessamento', preprocessador),
    ]
)

# Gerar o objeto do pipeline
pipeline.fit(X_train, y_train)

In [0]:
X_train.head(5)

In [0]:
pipeline

In [0]:
import pandas as pd
pd.DataFrame(pipeline.transform(X_train))

In [0]:
import pandas as pd

pd.DataFrame( pipeline.transform(X_train) ).describe().T